# Hotel Reservation Cancellation and ADR

## Objective
Harmonize three hotel-booking sources (160,665 raw bookings spanning 2015–2018), analyze cancellation patterns, and interpret time-aware classification and ADR regression results. These historical records are not current market data.

The notebook uses the same source mapping as training and the saved `models/metrics_summary.json`. Candidates fit on 2015–2016, model selection uses 2017 validation, and the 2018 holdout remains separate. Validation accuracy is not presented as future performance.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.datasets import load_merged_hotel_reservations

DATA_DIR = Path.cwd() / "data"
METRICS_PATH = Path.cwd() / "models" / "metrics_summary.json"
assert METRICS_PATH.is_file(), f"Metrics not found: {METRICS_PATH}"

bookings, source_audit = load_merged_hotel_reservations(DATA_DIR)
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))

print(f"Raw source rows: {source_audit['source_rows']:,}")
print(f"Schema-valid bookings: {len(bookings):,}")
print(f"Exact duplicate rows retained: {source_audit['exact_duplicate_rows_retained']:,}")
for source_name, details in source_audit['sources'].items():
    print(f"{details['file']}: {details['source_rows']:,} rows -> {details['usable_rows']:,} usable")
display(
    bookings.groupby("source_dataset", as_index=False)
    .agg(
        bookings=("is_canceled", "size"),
        cancellation_rate=("is_canceled", "mean"),
        average_adr=("avg_price_per_room", "mean"),
    )
)
print(f"Cancellation rate: {bookings['is_canceled'].mean():.2%}")
print(f"Average ADR: ${bookings.loc[bookings['is_canceled'].eq(0), 'avg_price_per_room'].mean():.2f}")
bookings.head()

## Data quality and shared preprocessing
The shared loader maps all three CSV schemas to common booking-time fields, normalizes month names and cancellation targets, and records the originating file. Legacy child and baby counts are combined because the newer reservations source has no separate babies field. Post-outcome status/date columns are excluded to avoid leakage.

In [ ]:
year_summary = (
    bookings.groupby("arrival_year", as_index=False)
    .agg(
        bookings=("is_canceled", "size"),
        cancellation_rate=("is_canceled", "mean"),
        average_adr=("avg_price_per_room", "mean"),
    )
)
year_summary["cancellation_rate"] = year_summary["cancellation_rate"].map(lambda value: f"{value:.2%}")
year_summary["average_adr"] = year_summary["average_adr"].map(lambda value: f"${value:.2f}")
year_summary

The merge includes **160,665 raw rows** and retains **31,994 repeated exact rows** from the full legacy file because there is no booking ID to prove that they are accidental duplicates. Rows with no guests or a negative ADR are excluded. The first cell reports per-source counts so these choices remain auditable.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#4C78A8", "#F58518", "#54A24B", "#E45756"]
bars = ax.bar(
    year_summary["arrival_year"].astype(str),
    bookings.groupby("arrival_year")["is_canceled"].mean().values * 100,
    color=colors,
)
ax.set_title("Cancellation rate by arrival year")
ax.set_xlabel("Arrival year")
ax.set_ylabel("Canceled bookings (%)")
ax.set_ylim(0, 45)
ax.bar_label(bars, fmt="%.1f%%", padding=3)
fig.tight_layout()
plt.show()

Cancellation patterns vary across source populations and years. Candidate models are fit on 2015–2016, selected on 2017, and evaluated on the independent 2018 source holdout; this keeps the final year out of preprocessing/model selection and reflects cross-source as well as temporal shift.

In [ ]:
classification_results = pd.DataFrame(metrics["classification"])
display(
    classification_results[
        ["Model", "Validation_Accuracy", "Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"]
    ].round(4)
)

selected_classifier = classification_results.loc[
    classification_results["Model"].eq(metrics["best_cls_model"])
].iloc[0]
print(f"Selected on 2017 validation: {metrics['best_cls_model']}")
print(f"2017 validation threshold: {metrics['classification_threshold']:.4f}")
print(f"2018 holdout accuracy: {selected_classifier['Accuracy']:.2%}")

confusion = np.asarray(selected_classifier["Confusion_Matrix"])
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(confusion, cmap="Blues")
ax.set(
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=["Not canceled", "Canceled"],
    yticklabels=["Not canceled", "Canceled"],
    xlabel="Predicted label",
    ylabel="Actual label",
    title="2018 holdout confusion matrix",
)
for row in range(confusion.shape[0]):
    for column in range(confusion.shape[1]):
        ax.text(column, row, f"{confusion[row, column]:,}", ha="center", va="center")
fig.colorbar(image, ax=ax, label="Bookings")
fig.tight_layout()
plt.show()

The classifier selected by 2017 validation accuracy was Random Forest (74.01% validation accuracy; threshold 0.5029). On the untouched 2018 holdout it achieved **67.34% accuracy**, **71.42% precision**, **18.39% recall**, and **77.03% ROC-AUC**. The low recall shows that its accuracy does not mean it catches most cancellations. Hist Gradient Boosting reached 81.17% holdout ROC-AUC but was not selected because the pre-declared selection metric is validation accuracy; do not choose a model based on the holdout.

In [ ]:
regression_results = pd.DataFrame(metrics["regression"])
display(regression_results.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    regression_results["Model"],
    regression_results["MAE ($)"],
    color=["#4C78A8", "#F58518", "#54A24B"],
)
ax.set_title("ADR mean absolute error on 2018 holdout")
ax.set_ylabel("MAE ($)")
ax.tick_params(axis="x", labelrotation=15)
ax.bar_label(bars, fmt="$%.2f", padding=3)
fig.tight_layout()
plt.show()

### Q&A
- **Were all three datasets combined?** Yes. The harmonized set contains 160,482 usable rows from 160,665 raw rows across three unique source files; the root-level duplicate file is not counted again.
- **Did cancellation accuracy stay high on the 2018 holdout?** No. The selected model achieved 67.34% accuracy and only 18.39% recall.
- **Can the ADR model be trusted as a pricing optimizer?** No. The model selected on 2017 validation achieved $22.19 MAE and 0.331 R² on the separate 2018 holdout.

### Data Analysis Key Findings
- Three schemas were mapped to shared booking fields, producing 160,482 usable rows; 183 invalid rows were excluded. The 31,994 repeated exact rows in the full legacy file were retained because no booking IDs prove they are accidental duplicates.
- Combined cancellation rates were 36.92% in 2015, 35.84% in 2016, 35.35% in 2017, and 36.71% in 2018.
- The selected cancellation model scored 67.34% accuracy, 71.42% precision, 18.39% recall, and 77.03% ROC-AUC on 2018.
- The ADR model selected on 2017 validation scored $22.19 MAE, $29.53 RMSE, and 0.331 R² on 2018.

### Insights or Next Steps
- Treat these results as historical evidence of temporal drift, not as a guarantee for a current hotel or future period.
- Validate and recalibrate with recent, hotel-specific bookings before operational use; keep pricing and booking decisions under human review.